# Revised experiments — JCLEPRO-D-26-17235 (Revision 1)
### Open-Data AI Framework for Climate Governance: Independent Validation of National Fossil-CO₂ Trajectories

Self-contained notebook: all code is in the cells below (no external modules). Running it top to bottom
reproduces every table and figure of the revised manuscript and of the response to reviewers
(about 30–40 minutes on one CPU core). Outputs are written to `out/`.

**Inputs expected in `data/`**
* `V0.6_Input_Data.xlsx` — the authors' compiled indicator file (unchanged; corrections are applied in code and logged).
* `owid_co2_india_GCB2025.csv` — fossil CO₂ for India, Global Carbon Budget 2025 via Our World in Data (Mt).
* `owid_energy_india.csv` — Ember / Energy Institute electricity series via Our World in Data (sensitivity run only).

**What changed relative to the original notebook** (reviewer comment in brackets)

| # | Change |
|---|---|
| 1 | Data audit: zero-coded missing values, terminal carry-forwards and rounded placeholders set to missing; target taken from the Global Carbon Budget 2025 (the original 2024 entry was 2023 + exactly 40 Mt); units are tonnes, reported in Mt [R1-1, R1-2] |
| 2 | Leakage-free nested pipeline: scaling, imputation, ranking, *k* and hyper-parameters fitted inside time-ordered training folds only; shuffled `KFold` removed; ensemble weights no longer use test R² [R2-1, R2-2] |
| 3 | Rolling-origin (19 origins, horizons 1–6) and retrospective ex-ante forecasting [R2-2, R2-3, R4-3] |
| 4 | Statistical benchmarks (random walk with drift, ARIMA, ETS, STIRPAT, ARIMAX), MASE and skill scores [R2-3, R2-4] |
| 5 | Identical tuning for all 11 learners with a standardised target (the original LSTM/GRU/DNN produced constant predictions because the ~10⁹ target was unscaled); stationary-target experiment [R2-4, R4-4] |
| 6 | Post-hoc "transition smoothing" of the forecast removed; empirical prediction intervals; Kaya scenarios [R2-6] |
| 7 | Collinearity-robust importance (VIF, bootstrap coefficients, dominance analysis, grouped permutation) and LMDI decomposition; causal language dropped [R2-5, R4-1] |
| 8 | Structural-break, cointegration and emission-factor sensitivity analyses [R4-3, R4-5, R2-3] |
| 9 | Data-availability and data-quality experiments [R1-1, R1-2] |

Random seed 42 throughout.

## 0. Set-up — Google Drive + Colab

**One-off set-up (5 minutes)**

1. In Google Drive, create a folder named **`JCLEPRO_revision`** in *My Drive*.
2. Unzip `revised_code.zip` on your computer and upload into that folder:
   * the whole **`data`** folder (3 files: `V0.6_Input_Data.xlsx`, `owid_co2_india_GCB2025.csv`, `owid_energy_india.csv`);
   * this notebook.
   So Drive looks like this:

```
My Drive/
└── JCLEPRO_revision/
    ├── Revised_Code_Colab_JCLEPRO-D-26-17235.ipynb
    ├── data/
    │   ├── V0.6_Input_Data.xlsx
    │   ├── owid_co2_india_GCB2025.csv
    │   └── owid_energy_india.csv
    └── out/          ← created automatically; all tables and figures land here
```
3. Right-click the notebook in Drive → **Open with → Google Colaboratory**.
4. In Colab: **Runtime → Change runtime type → CPU** (no GPU is needed; GPU
   runtimes are not faster here and are rationed).
5. Run the two cells below. The first mounts Drive and checks the folder; it will
   ask you to authorise Drive access once per session.

**Then run the cells in order.** Everything a later stage needs is written to
`out/` on Drive, so progress survives a disconnect: after reconnecting, re-run
Section 0 and Section 1 (a few seconds) and then only the stages the status
table still shows as "missing".

| Section | What it does | Time (Colab CPU) |
|---|---|---|
| 0 Set-up | mount Drive, check files, install packages | 1–2 min (first run) |
| 1 Library | define all functions | seconds |
| 2 Stage 1 | hold-out evaluation of 11 learners + benchmarks | 3–5 min |
| 3 Stage 2 | 19 rolling origins — **the long one** | 10–15 min |
| 4 Stage 3 | stationary-target experiment | 2–4 min |
| 5 Stage 4 | projection, intervals, scenarios | 1–2 min |
| 6 Stage 5 | importance, LMDI, break and cointegration tests | 2–4 min |
| 7 Stage 6 | data-availability and data-quality experiments | 10–15 min |
| 8 Stage 7 | figures | under 1 min |
| 9 Download | zip `out/` | seconds |

Total ≈ 35–50 minutes. Colab disconnects idle sessions, so keep the browser tab
open while Sections 3 and 7 run, or run them one at a time and let `out/` on
Drive hold the results in between.

In [ ]:
# === Section 0.1 — mount Drive, set the project folder, check the input files ========
# The ONLY line you may need to edit is PROJECT, if you named the Drive folder
# something else or placed it inside another folder.
import os, sys, shutil, importlib.util

PROJECT = "/content/drive/MyDrive/JCLEPRO_revision"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    if not os.path.isdir(PROJECT):
        raise SystemExit(f"Folder not found: {PROJECT}\n"
                         "Create it in My Drive (or edit PROJECT above) and re-run this cell.")
else:
    PROJECT = os.getcwd()          # running locally: use the notebook's own folder

os.makedirs(f"{PROJECT}/data", exist_ok=True)
os.makedirs(f"{PROJECT}/out", exist_ok=True)
os.chdir(PROJECT)

# Input files (relative paths — every later cell uses these three names)
XLSX   = "data/V0.6_Input_Data.xlsx"
GCB    = "data/owid_co2_india_GCB2025.csv"
ENERGY = "data/owid_energy_india.csv"

def _find_inputs():
    return [p for p in (XLSX, GCB, ENERGY) if not os.path.exists(p)]

missing = _find_inputs()
if missing and IN_COLAB:
    # Fallback: let the browser upload them, then file them into data/ under the right names
    from google.colab import files
    print("Missing from data/:", missing)
    print("Select V0.6_Input_Data.xlsx, owid_co2_india_GCB2025.csv and owid_energy_india.csv ...")
    for name in files.upload():
        low = name.lower()
        dest = XLSX if low.endswith((".xlsx", ".xls")) else (ENERGY if "energy" in low else GCB)
        shutil.move(name, dest)
        print(f"  {name}  ->  {dest}")
    missing = _find_inputs()
if missing:
    raise SystemExit(f"Still missing: {missing}\nPut these files in {PROJECT}/data/ and re-run.")

print("Working folder :", os.getcwd())
print("Input files    : all three present")


Mounted at /content/drive
Working folder : /content/drive/MyDrive/JCLEPRO_revision
Input files    : all three present


In [ ]:
# === Section 0.2 — packages, sanity check of the inputs, and a progress table =======
import importlib, subprocess, sys

need = []
for mod, spec in [("sklearn", "scikit-learn"), ("statsmodels", "statsmodels"), ("xgboost", "xgboost"),
                  ("lightgbm", "lightgbm"), ("catboost", "catboost"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(mod) is None:
        need.append(spec)
if need:
    print("Installing:", ", ".join(need), "(1-2 minutes; only needed once per Colab session)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Done. If Colab asks you to restart the runtime, do so and re-run Section 0 only.")

import pandas as pd, numpy as np, sklearn, statsmodels
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | statsmodels {statsmodels.__version__}")

_x = pd.read_excel(XLSX)
_g = pd.read_csv(GCB)
print(f"Indicator file : {_x.shape[0]} rows x {_x.shape[1]} columns, years {_x.Year.min()}-{_x.Year.max()}")
print(f"GCB CO2 file   : years {_g.year.min()}-{_g.year.max()}, 2024 = {_g.set_index('year').co2.get(2024):.1f} Mt")

# ---- progress table: which stages have already produced their outputs --------------
STAGES = {
    "2  Stage 1  hold-out evaluation":        ["out/stage1.pkl", "out/Table2_holdout_all_models.csv"],
    "3  Stage 2  rolling origins":            ["out/rolling_origin_forecasts.csv", "out/Table3_rolling_origin_by_horizon.csv"],
    "4  Stage 3  stationary target":          ["out/Table_S4b_stationary_target_experiment.csv"],
    "5  Stage 4  projection and scenarios":   ["out/stage4.pkl", "out/Table5_scenarios.csv"],
    "6  Stage 5  importance and diagnostics": ["out/stage5.pkl", "out/Table7_LMDI_Kaya_MtCO2.csv"],
    "7  Stage 6  data experiments":           ["out/Table_S7a_data_experiments_deterministic.csv",
                                                "out/Table_S7b_data_quality_montecarlo.csv"],
    "8  Stage 7  figures":                    ["out/Figure5_forecast_intensity_scenarios.png"],
}

def status():
    """Print which sections still need to be run (useful after a Colab disconnect)."""
    rows = [{"section": k, "status": "done" if all(os.path.exists(f) for f in v) else "to run"}
            for k, v in STAGES.items()]
    display(pd.DataFrame(rows).set_index("section"))

try:
    from IPython.display import display, Image      # notebook display helpers
except ModuleNotFoundError:                          # plain-python run
    display, Image = print, None
status()


Installing: catboost (1-2 minutes; only needed once per Colab session)
Done. If Colab asks you to restart the runtime, do so and re-run Section 0 only.
pandas 2.2.3 | numpy 2.1.3 | scikit-learn 1.6.1 | statsmodels 0.15.0
Indicator file : 60 rows x 24 columns, years 1965-2024
GCB CO2 file   : years 1850-2024, 2024 = 3193.5 Mt


,status
section,
2 Stage 1 hold-out evaluation,done
3 Stage 2 rolling origins,done
4 Stage 3 stationary target,done
5 Stage 4 projection and scenarios,done
6 Stage 5 importance and diagnostics,done
7 Stage 6 data experiments,done
8 Stage 7 figures,done


In [ ]:
# === Section 0.3 — display options =================================================
import os, pickle, itertools, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
assert os.path.basename(os.getcwd()) and os.path.exists(XLSX), "Run Section 0.1 first."


## 1. Library

### 1.1 Imports, seed and constants

In [ ]:
import warnings, itertools, time
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
SEED = 42


### 1.2 Data audit and cleaning  [R1-1, R1-2]
Zero placeholders, terminal carry-forwards and individually documented suspect entries are set to missing;
the target is the Global Carbon Budget 2025 fossil-CO₂ series in tonnes. Target-derived features
(emission intensity, emissions per capita) are **not** created, because they would leak the target.

In [ ]:
# ----------------------------------------------------------------------------
# 1. DATA AUDIT AND CLEANING
# ----------------------------------------------------------------------------
TARGET = "CO2_t"          # fossil + industry CO2, tonnes (NOT kt; see audit)
YEAR = "Year"
RAW_TARGET = "CO2 Emissions total (kt)"   # original (mis-labelled) column name

# Features whose zeros are placeholders for "not available"
ZERO_AS_MISSING = [
    "Primary Energy Production (million metric tons of oil eq.)",
    "Electricity generation - TWh",
    "Electric Power Consumption (kWh per capita)",
    "Access to Clean Cooking Fuels",
    "Electricity from fossil fuels - TWh",
    "Human Development Index",
    "Air Quality Index",
]
# Individually documented suspect entries (placeholders / rounded / inconsistent)
SUSPECT_VALUES = {
    ("Energy intensity(Kwh per dollar)", 2023): "rounded placeholder (0.95) inconsistent with series precision",
    ("Energy intensity(Kwh per dollar)", 2024): "rounded placeholder (0.94) inconsistent with series precision",
    ("Energy use (KG of oil equivalent per capita)", 2024): "rounded placeholder (730.0)",
    ("Human Development Index", 2024): "0.644 inconsistent with UNDP series (0.685 in 2023); no 2024 release",
}

SECTOR = {  # sector mapping used for grouped importance (Section 3.6)
    "GDP (Constant LCU)": "Economic", "GDP growth rate": "Economic",
    "Exports of goods and services (Current LCU)": "Economic", "Import (Current LCU)": "Economic",
    "Industrial Production Manufacturing value added (Current US Dollar)": "Economic",
    "Energy intensity(Kwh per dollar)": "Energy",
    "Primary Energy Production (million metric tons of oil eq.)": "Energy",
    "Primary energy consumption (TWH)": "Energy",
    "Fossil fuels (% equivalent primary energy)": "Energy",
    "Energy use (KG of oil equivalent per capita)": "Energy",
    "Electricity from renewables - TWh": "Energy", "Electricity generation - TWh": "Energy",
    "Electric Power Consumption (kWh per capita)": "Energy",
    "Electricity from fossil fuels - TWh": "Energy", "Renewable_Share": "Energy",
    "Energy_Efficiency": "Energy",
    "Population": "Social", "Population Growth Rate Annual": "Social",
    "Human Development Index": "Social", "Education Literacy Rates": "Social",
    "Urbanization Rate": "Social", "Access to Clean Cooking Fuels": "Social",
    "Forest Area": "Environmental", "Air Quality Index": "Environmental",
}
SHORT = {
    "GDP (Constant LCU)": "GDP", "Electric Power Consumption (kWh per capita)": "Elec. cons. p.c.",
    "Urbanization Rate": "Urbanisation", "Electricity from fossil fuels - TWh": "Fossil electricity",
    "Population": "Population", "Primary energy consumption (TWH)": "Primary energy",
    "Energy use (KG of oil equivalent per capita)": "Energy use p.c.",
    "Electricity generation - TWh": "Elec. generation", "Import (Current LCU)": "Imports",
    "Exports of goods and services (Current LCU)": "Exports",
    "Industrial Production Manufacturing value added (Current US Dollar)": "Manufacturing VA",
    "Primary Energy Production (million metric tons of oil eq.)": "Primary energy prod.",
    "Human Development Index": "HDI", "Education Literacy Rates": "Literacy",
    "Electricity from renewables - TWh": "Renewable electricity", "Forest Area": "Forest area",
    "Access to Clean Cooking Fuels": "Clean cooking", "Air Quality Index": "AQI",
    "Population Growth Rate Annual": "Pop. growth", "GDP growth rate": "GDP growth",
    "Energy intensity(Kwh per dollar)": "Energy intensity", "Renewable_Share": "Renewable share",
    "Energy_Efficiency": "Energy efficiency", "Fossil fuels (% equivalent primary energy)": "Fossil share",
}


def terminal_carry_forward(s):
    """Indices of a terminal run of identical (non-missing) values, excluding the first of the run."""
    vals = s.values
    idx = []
    i = len(vals) - 1
    while i > 0 and pd.notna(vals[i]) and vals[i] == vals[i - 1]:
        idx.append(s.index[i]); i -= 1
    return sorted(idx)


def load_and_audit(xlsx_path, gcb_csv=None, use_gcb_target=True, zeros_as_missing=True,
                   drop_suspect=True):
    """Return cleaned DataFrame (indexed by year) and an audit log DataFrame."""
    raw = pd.read_excel(xlsx_path).set_index(YEAR)
    df = raw.copy()
    log = []
    # target
    df = df.rename(columns={RAW_TARGET: "CO2_original_t"})
    if use_gcb_target and gcb_csv is not None:
        g = pd.read_csv(gcb_csv).set_index("year")["co2"] * 1e6  # Mt -> t
        df[TARGET] = g.reindex(df.index)
        diff = (df["CO2_original_t"] - df[TARGET]) / df[TARGET] * 100
        for y in df.index:
            if abs(diff[y]) > 0.5:
                log.append(dict(variable="CO2 (target)", year=y, original=df.loc[y, "CO2_original_t"],
                                action="replaced by Global Carbon Budget 2025 value",
                                reason=f"differs by {diff[y]:+.2f}% from GCB 2025 (original 2024 value = 2023 + exactly 40 Mt)"))
    else:
        df[TARGET] = df["CO2_original_t"]
    if zeros_as_missing:
        for c in ZERO_AS_MISSING:
            z = df.index[df[c] == 0]
            if len(z):
                log.append(dict(variable=c, year=f"{z.min()}-{z.max()}", original=0,
                                action="set to missing", reason=f"{len(z)} zero placeholders (series not yet available)"))
                df.loc[z, c] = np.nan
        for c in [c for c in df.columns if c not in (TARGET, "CO2_original_t")]:
            cf = terminal_carry_forward(df[c])
            if cf:
                log.append(dict(variable=c, year=f"{cf[0]}-{cf[-1]}", original=df.loc[cf[0], c],
                                action="set to missing", reason="terminal carry-forward of last observed value"))
                df.loc[cf, c] = np.nan
    if drop_suspect:
        for (c, y), why in SUSPECT_VALUES.items():
            if pd.notna(df.loc[y, c]):
                log.append(dict(variable=c, year=y, original=df.loc[y, c], action="set to missing", reason=why))
                df.loc[y, c] = np.nan
    # derived features (no target-derived features -> no leakage)
    df["Renewable_Share"] = df["Electricity from renewables - TWh"] / df["Electricity generation - TWh"] * 100
    df["Energy_Efficiency"] = df["GDP (Constant LCU)"] / df["Primary energy consumption (TWH)"]
    return df, pd.DataFrame(log)


def candidate_features(df):
    return [c for c in df.columns if c not in (TARGET, "CO2_original_t")]


### 1.3 Leakage-free pipeline: preparation, coverage rule, consensus ranking, nested tuning  [R2-1, R2-2]
`Prep` (NaN-aware scaling → iterative imputation), the eligibility rule (a candidate must be observed in at
least half of the training window), the triangulated ranking (mutual information, Random-Forest impurity,
permutation importance) and the inner `TimeSeriesSplit` search over *k* and hyper-parameters. Every one of
these steps is refitted inside each training fold; the target is standardised for all learners.

In [ ]:
# ----------------------------------------------------------------------------
# 2. PIPELINE BUILDING BLOCKS (fit on training data only)
# ----------------------------------------------------------------------------
class Prep:
    """Standardise (NaN-aware) -> iterative imputation. Fitted on training rows only."""
    def fit(self, X):
        self.cols = list(X.columns)
        self.sc = StandardScaler().fit(X.values)
        self.sc.scale_[~np.isfinite(self.sc.scale_) | (self.sc.scale_ == 0)] = 1.0
        self.sc.mean_[~np.isfinite(self.sc.mean_)] = 0.0
        Z = self.sc.transform(X.values)
        self.imp = IterativeImputer(estimator=BayesianRidge(), max_iter=25, random_state=SEED,
                                    keep_empty_features=True, skip_complete=True).fit(Z)
        return self

    def transform(self, X):
        Z = self.imp.transform(self.sc.transform(X[self.cols].values))
        return pd.DataFrame(Z, index=X.index, columns=self.cols)

    def inverse(self, Z):
        return pd.DataFrame(self.sc.inverse_transform(Z.values), index=Z.index, columns=self.cols)


MIN_COVERAGE = 0.5  # candidate must be observed in >=50% of the training window


def eligible(X, min_cov=MIN_COVERAGE):
    """Features observed in at least `min_cov` of the training rows (computed on training data only).
    Heavily imputed series would otherwise act as synthetic combinations of other predictors."""
    return [c for c in X.columns if X[c].notna().mean() >= min_cov]


def consensus_ranking(Z, y, seed=SEED):
    """Triangulated ranking: mutual information, RF impurity and RF permutation importance
    (mean of ranks). Computed on training data only."""
    mi = mutual_info_regression(Z.values, y, random_state=seed)
    rf = RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=1).fit(Z.values, y)
    pi = permutation_importance(rf, Z.values, y, n_repeats=10, random_state=seed, n_jobs=1).importances_mean
    imp = pd.DataFrame({"MI": mi, "RF_MDI": rf.feature_importances_, "Permutation": pi}, index=Z.columns)
    ranks = imp.rank(ascending=False, method="average")
    imp["mean_rank"] = ranks.mean(axis=1)
    return imp.sort_values("mean_rank")


def wrap(model):
    """Scale the target for every learner (critical for SVR / MLP; harmless for linear)."""
    return TransformedTargetRegressor(regressor=model, transformer=StandardScaler())


def model_space(families=None, fast=False):
    import xgboost as xgb, lightgbm as lgb
    from catboost import CatBoostRegressor
    sp = {}
    sp["OLS"] = [LinearRegression()]
    sp["Ridge"] = [Ridge(alpha=a) for a in np.logspace(-4, 2, 13)]
    sp["Lasso"] = [Lasso(alpha=a, max_iter=50000) for a in np.logspace(-5, 0, 11)]
    sp["ElasticNet"] = [ElasticNet(alpha=a, l1_ratio=l, max_iter=50000)
                        for a in np.logspace(-5, 0, 6) for l in (0.2, 0.5, 0.8)]
    if not fast:
        sp["RandomForest"] = [RandomForestRegressor(n_estimators=150, max_depth=d, max_features=m,
                              random_state=SEED, n_jobs=1) for d in (3, None) for m in (1.0, 0.5)]
        sp["ExtraTrees"] = [ExtraTreesRegressor(n_estimators=150, max_depth=d, max_features=m,
                            random_state=SEED, n_jobs=1) for d in (3, None) for m in (1.0, 0.5)]
        sp["GradientBoosting"] = [GradientBoostingRegressor(n_estimators=200, learning_rate=lr, max_depth=d,
                                  subsample=1.0, random_state=SEED) for lr in (0.05, 0.1) for d in (2, 3)]
        sp["XGBoost"] = [xgb.XGBRegressor(n_estimators=200, learning_rate=lr, max_depth=d, subsample=0.8,
                         reg_lambda=1.0, random_state=SEED, n_jobs=1, verbosity=0)
                         for lr in (0.05, 0.1) for d in (2, 3)]
        sp["LightGBM"] = [lgb.LGBMRegressor(n_estimators=n, learning_rate=0.05, num_leaves=l, min_child_samples=3,
                          min_data_in_bin=1, random_state=SEED, n_jobs=1, verbose=-1)
                          for n in (100, 300) for l in (4, 8)]
        sp["CatBoost"] = [CatBoostRegressor(iterations=300, learning_rate=lr, depth=d, random_seed=SEED,
                          verbose=0, thread_count=1) for lr in (0.05, 0.1) for d in (2, 4)]
        sp["SVR"] = [SVR(kernel="rbf", C=C, gamma=g, epsilon=e)
                     for C in (1, 10, 100) for g in ("scale", 0.01, 0.1) for e in (0.01, 0.1)]
        sp["MLP"] = [MLPRegressor(hidden_layer_sizes=h, alpha=a, solver="lbfgs", max_iter=5000, random_state=SEED)
                     for h in ((16,), (32, 16)) for a in (1e-3, 1e-1)]
    if families:
        sp = {k: v for k, v in sp.items() if k in families}
    return sp


K_GRID = [3, 4, 5, 6, 8, 10, 12, 15, 18]
ALL = 99  # code for "all eligible candidate features" (no selection)


def describe(m):
    p = m.get_params()
    keys = ["alpha", "l1_ratio", "max_depth", "max_features", "learning_rate", "n_estimators",
            "num_leaves", "depth", "C", "gamma", "epsilon", "hidden_layer_sizes"]
    return ", ".join(f"{k}={p[k]}" for k in keys if k in p and p[k] is not None)


def inner_folds(n):
    test_size = 5 if n >= 40 else 3
    n_splits = min(5, max(2, (n - 12) // test_size))
    return TimeSeriesSplit(n_splits=n_splits, test_size=test_size)


def tune(X, y, space, k_grid=K_GRID, fixed_features=None, score="mape"):
    """Nested inner time-series CV. For every inner fold: fit Prep and consensus ranking on the
    inner-training rows only; evaluate every (family, config, k). Returns table of mean MAPE."""
    n = len(y)
    rows = []
    for fold, (tr, va) in enumerate(inner_folds(n).split(X)):
        Xtr, Xva, ytr, yva = X.iloc[tr], X.iloc[va], y[tr], y[va]
        prep = Prep().fit(Xtr)
        Ztr, Zva = prep.transform(Xtr), prep.transform(Xva)
        if fixed_features is None:
            order = consensus_ranking(Ztr[eligible(Xtr)], ytr).index.tolist()
            ks = [k for k in k_grid if k < len(order)] + [ALL]
        else:
            order, ks = list(fixed_features), [len(fixed_features)]
        for fam, configs in space.items():
            for ci, cfg in enumerate(configs):
                for k in ks:
                    cols = order[:k] if k != ALL else order
                    try:
                        m = wrap(clone(cfg)).fit(Ztr[cols].values, ytr)
                        p = m.predict(Zva[cols].values)
                        ape = np.abs((yva - p) / yva) if score == "mape" else np.abs(yva - p) / np.std(y)
                    except Exception:
                        ape = np.full(len(va), np.inf)
                    rows.extend(dict(family=fam, cfg=ci, k=k, fold=fold, ape=a) for a in ape)
    r = pd.DataFrame(rows)
    fm = r.groupby(["family", "cfg", "k", "fold"])["ape"].mean().mul(100)
    g = fm.groupby(["family", "cfg", "k"])
    t = pd.DataFrame({"cv_mape": r.groupby(["family", "cfg", "k"])["ape"].mean().mul(100),
                      "cv_se": g.std() / np.sqrt(g.count())}).reset_index()
    return t


def best_by_family(tab, k_fixed=None, rule="min"):
    """rule='min': lowest inner-CV MAPE. rule='1se': most parsimonious k (then lowest MAPE)
    within one standard error of the minimum (Hastie et al., one-standard-error rule)."""
    t = tab if k_fixed is None else tab[tab.k == k_fixed]
    if rule == "min":
        return t.loc[t.groupby("family")["cv_mape"].idxmin()].set_index("family")
    out = []
    for fam, g in t.groupby("family"):
        b = g.loc[g.cv_mape.idxmin()]
        se = b.cv_se if np.isfinite(b.cv_se) else 0.0
        ok = g[g.cv_mape <= b.cv_mape + se + 1e-12]
        out.append(ok.sort_values(["k", "cv_mape"]).iloc[0])
    return pd.DataFrame(out).set_index("family")


class FittedModel:
    """Prep + ranking + top-k + (wrapped) estimator fitted on a training window."""
    def __init__(self, cfg, k, features=None):
        self.cfg, self.k, self.features = cfg, k, features

    def fit(self, X, y):
        self.prep = Prep().fit(X)
        Z = self.prep.transform(X)
        if self.features is None:
            self.ranking = consensus_ranking(Z[eligible(X)], y)
            self.cols = self.ranking.index[: self.k].tolist() if self.k != ALL else self.ranking.index.tolist()
        else:
            self.ranking, self.cols = None, list(self.features)
        self.model = wrap(clone(self.cfg)).fit(Z[self.cols].values, y)
        self.Z_train = Z
        return self

    def predict(self, X):
        Z = self.prep.transform(X)
        return self.model.predict(Z[self.cols].values)


### 1.4 Metrics (R², MAE, RMSE, MAPE, MASE)  [R2-3]

In [ ]:
# ----------------------------------------------------------------------------
# 3. METRICS
# ----------------------------------------------------------------------------
def metrics(y, p, y_train=None):
    y, p = np.asarray(y, float), np.asarray(p, float)
    out = dict(R2=r2_score(y, p), MAE=mean_absolute_error(y, p), RMSE=np.sqrt(mean_squared_error(y, p)),
               MAPE=np.mean(np.abs((y - p) / y)) * 100)
    if y_train is not None:
        scale = np.mean(np.abs(np.diff(np.asarray(y_train, float))))
        out["MASE"] = out["MAE"] / scale
    return out


### 1.5 Predictor projection for ex-ante forecasting  [R2-2, R2-6]

In [ ]:
# ----------------------------------------------------------------------------
# 4. FEATURE PROJECTION (ex-ante forecasting)
# ----------------------------------------------------------------------------
def project_features(X_hist, horizon, window=10, method="loglinear"):
    """Project each (imputed, original-unit) feature `horizon` years ahead using only X_hist.
    loglinear: OLS trend on log values over the last `window` years (constant growth) for
    strictly positive series; linear trend otherwise. quadratic: original manuscript method."""
    last = X_hist.index.max()
    fut_idx = np.arange(last + 1, last + horizon + 1)
    out = {}
    for c in X_hist.columns:
        s = X_hist[c].dropna().iloc[-window:] if method != "quadratic" else X_hist[c].dropna().iloc[-30:]
        t = s.index.values.astype(float)
        if method == "quadratic":
            A = np.column_stack([np.ones_like(t), t - t[0], (t - t[0]) ** 2])
            b = np.linalg.lstsq(A, s.values, rcond=None)[0]
            tf = fut_idx - t[0]
            out[c] = b[0] + b[1] * tf + b[2] * tf ** 2
        elif (s > 0).all():
            b = np.polyfit(t, np.log(s.values), 1)
            out[c] = np.exp(np.polyval(b, fut_idx))
        else:
            b = np.polyfit(t, s.values, 1)
            out[c] = np.polyval(b, fut_idx)
    return pd.DataFrame(out, index=fut_idx)


def imputed_history(prep, X):
    """Original-unit feature history with missing values filled by the fitted imputer."""
    return prep.inverse(prep.transform(X))


### 1.6 Statistical benchmarks  [R2-4]

In [ ]:
# ----------------------------------------------------------------------------
# 5. STATISTICAL BENCHMARKS
# ----------------------------------------------------------------------------
def rw_drift(y_train, h):
    ly = np.log(y_train)
    d = np.mean(np.diff(ly))
    return np.exp(ly[-1] + d * np.arange(1, h + 1))


def arima_auto(y_train, h, exog_train=None, exog_future=None):
    from statsmodels.tsa.arima.model import ARIMA
    ly = np.log(y_train)
    best, best_aic = None, np.inf
    orders = [(p, 1, q) for p in (0, 1, 2) for q in (0, 1, 2)] if exog_train is None else [(1, 1, 0), (0, 1, 1), (0, 1, 0)]
    for o in orders:
        try:
            r = ARIMA(ly, order=o, exog=exog_train, trend="t" if exog_train is None else "n").fit()
            if r.aic < best_aic:
                best, best_aic = r, r.aic
        except Exception:
            pass
    return np.exp(best.forecast(h, exog=exog_future))


def ets_holt(y_train, h):
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    r = ExponentialSmoothing(np.log(y_train), trend="add", damped_trend=False).fit()
    return np.exp(r.forecast(h))


def stirpat_design(X):
    """ln CO2 = a + b ln P + c ln(GDP/P) + d ln(E/GDP) + e ln(fossil share)"""
    P = X["Population"]; G = X["GDP (Constant LCU)"]; E = X["Primary energy consumption (TWH)"]
    F = X["Fossil fuels (% equivalent primary energy)"]
    return np.column_stack([np.ones(len(X)), np.log(P), np.log(G / P), np.log(E / G), np.log(F)])


def stirpat_fit_predict(Xtr, ytr, Xte):
    b = np.linalg.lstsq(stirpat_design(Xtr), np.log(ytr), rcond=None)[0]
    return np.exp(stirpat_design(Xte) @ b)


### 1.7 Intensity benchmark, collinearity-robust importance and LMDI  [R2-5, R4-1]

In [ ]:
# ----------------------------------------------------------------------------
# 6. NDC INTENSITY BENCHMARK
# ----------------------------------------------------------------------------
def intensity_reduction(co2, gdp, base_co2, base_gdp):
    return (1 - (co2 / gdp) / (base_co2 / base_gdp)) * 100

# ----------------------------------------------------------------------------
# 7. IMPORTANCE UNDER COLLINEARITY
# ----------------------------------------------------------------------------
def vif(Z):
    out = {}
    for c in Z.columns:
        others = [o for o in Z.columns if o != c]
        r2 = LinearRegression().fit(Z[others], Z[c]).score(Z[others], Z[c])
        out[c] = 1 / max(1 - r2, 1e-12)
    return pd.Series(out)


def shapley_r2(Z, y, groups=None):
    """Dominance analysis (general dominance / Shapley decomposition of OLS R^2).
    groups: dict name -> list of columns (group-level decomposition)."""
    from math import factorial
    units = groups if groups is not None else {c: [c] for c in Z.columns}
    names = list(units)
    n = len(names)
    cache = {}

    def r2(S):
        key = frozenset(S)
        if key not in cache:
            if not S:
                cache[key] = 0.0
            else:
                cols = [c for s in S for c in units[s]]
                cache[key] = LinearRegression().fit(Z[cols], y).score(Z[cols], y)
        return cache[key]
    phi = {}
    for i in names:
        others = [j for j in names if j != i]
        tot = 0.0
        for r in range(len(others) + 1):
            w = factorial(r) * factorial(n - r - 1) / factorial(n)
            for S in itertools.combinations(others, r):
                tot += w * (r2(S + (i,)) - r2(S))
        phi[i] = tot
    return pd.Series(phi), r2(tuple(names))


def lmdi_kaya(co2, pop, gdp, energy, y0, y1):
    """Additive LMDI-I decomposition of the change in CO2 between y0 and y1 (same units as co2)."""
    f = {"Population": pop, "Affluence (GDP per capita)": gdp / pop,
         "Energy intensity (E/GDP)": energy / gdp, "Carbon intensity of energy (CO2/E)": co2 / energy}
    C0, C1 = co2[y0], co2[y1]
    L = (C1 - C0) / (np.log(C1) - np.log(C0))
    res = {k: L * np.log(v[y1] / v[y0]) for k, v in f.items()}
    res["Total change"] = C1 - C0
    return pd.Series(res)


## 2. Stage 1 — Data audit and leakage-free hold-out evaluation
Training 1965–2012, test 2013–2024 (used once). All 11 learners plus five benchmarks; the
'selected' rows use the inner-CV choice of *k*, 'selected_1se' the one-standard-error rule, 'all' no selection.
Runtime ≈ 2–3 min.

In [ ]:
import pickle, time, numpy as np, pandas as pd

df, audit = load_and_audit(XLSX, GCB)
audit.to_csv("out/Table_S1_data_audit.csv", index=False)
FEATS = candidate_features(df)
TR, TE = slice(1965, 2012), slice(2013, 2024)
Xtr, ytr = df.loc[TR, FEATS], df.loc[TR, TARGET].values
Xte, yte = df.loc[TE, FEATS], df.loc[TE, TARGET].values

t0 = time.time()
space = model_space()
tab = tune(Xtr, ytr, space)
print("tuning time", round(time.time() - t0, 1))
best_sel = best_by_family(tab)
best_all = best_by_family(tab, k_fixed=ALL)
best_1se = best_by_family(tab, rule="1se")

rows, preds, fitted = [], {}, {}
for fam in space:
    for tag, b in (("selected", best_sel.loc[fam]), ("selected_1se", best_1se.loc[fam]), ("all", best_all.loc[fam])):
        cfg = space[fam][int(b.cfg)]
        fm = FittedModel(cfg, int(b.k)).fit(Xtr, ytr)
        p_te, p_tr = fm.predict(Xte), fm.predict(Xtr)
        m = metrics(yte, p_te, ytr); mt = metrics(ytr, p_tr)
        rows.append(dict(family=fam, set=tag, k=len(fm.cols), params=describe(cfg), inner_cv_mape=b.cv_mape,
                         R2_train=mt["R2"], **{f"{kk}_test": v for kk, v in m.items()}))
        preds[(fam, tag)] = (p_tr, p_te)
        fitted[(fam, tag)] = fm
    print(fam, "done", round(time.time() - t0, 1))

# benchmarks (ex-ante by construction: they only use y up to 2012)
h = len(yte)
bench = {"RW-drift": rw_drift(ytr, h), "ARIMA": arima_auto(ytr, h), "ETS (Holt)": ets_holt(ytr, h)}
# STIRPAT and ARIMAX use actual regressors (ex-post), imputed with training-only Prep
prep = Prep().fit(Xtr)
Htr, Hte = imputed_history(prep, Xtr), imputed_history(prep, Xte)
bench["STIRPAT (ex-post)"] = stirpat_fit_predict(Htr, ytr, Hte)
ex_tr = np.log(Htr[["GDP (Constant LCU)", "Primary energy consumption (TWH)"]].values)
ex_te = np.log(Hte[["GDP (Constant LCU)", "Primary energy consumption (TWH)"]].values)
bench["ARIMAX (ex-post)"] = arima_auto(ytr, h, ex_tr, ex_te)
for b, p in bench.items():
    m = metrics(yte, p, ytr)
    rows.append(dict(family=b, set="benchmark", k=np.nan, params="", inner_cv_mape=np.nan, R2_train=np.nan,
                     **{f"{kk}_test": v for kk, v in m.items()}))
    preds[(b, "benchmark")] = (None, p)

res = pd.DataFrame(rows)
rw_mae = res.loc[res.family == "RW-drift", "MAE_test"].values[0]
res["Skill_vs_RW_%"] = (1 - res["MAE_test"] / rw_mae) * 100
res.to_csv("out/Table2_holdout_all_models.csv", index=False)
tab.to_csv("out/inner_cv_table_holdout.csv", index=False)

# stability of selection: ranking computed on the full training window
rank_tr = fitted[("Lasso", "selected")].ranking
rank_tr.to_csv("out/Table1_consensus_ranking_train1965_2012.csv")
pickle.dump(dict(res=res, preds=preds, best_sel=best_sel, best_all=best_all, tab=tab, rank_tr=rank_tr,
                 best_1se=best_1se, fitted={k: v for k, v in fitted.items() if k[0] in ("OLS", "Ridge", "Lasso", "ElasticNet")}),
            open("out/stage1.pkl", "wb"))
pd.set_option("display.width", 250)
print(res.round(4).to_string())
print("total", round(time.time() - t0, 1))


tuning time 244.3
OLS done 260.3
Ridge done 276.2
Lasso done 292.2
ElasticNet done 308.6
RandomForest done 326.3
ExtraTrees done 342.8
GradientBoosting done 359.1
XGBoost done 375.5
LightGBM done 391.4
CatBoost done 408.1
SVR done 424.1
MLP done 441.8


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


               family           set     k                                                                           params  inner_cv_mape  R2_train  R2_test      MAE_test     RMSE_test  MAPE_test  MASE_test  Skill_vs_RW_%
0                 OLS      selected  18.0                                                                                          1.4380    0.9997   0.9041  6.907335e+07  1.064447e+08     2.5971     1.8307        70.7346
1                 OLS  selected_1se  18.0                                                                                          1.4380    0.9997   0.9041  6.907335e+07  1.064447e+08     2.5971     1.8307        70.7346
2                 OLS           all  21.0                                                                                         16.3274    0.9997   0.8985  5.357659e+07  1.095275e+08     1.8805     1.4200        77.3004
3               Ridge      selected  18.0                                                                     al

## 3. Stage 2 — Rolling-origin evaluation (ex post and ex ante)  [R2-2, R4-3]
19 forecast origins (2000–2018), horizons 1–6 years; the entire pipeline is re-tuned at every origin.
Origins 2004, 2009, 2014 and 2018 are the retrospective six-year experiments. Runtime ≈ 8–10 min.

> **Longest cell (10–15 min).** Keep the Colab tab open. If the session drops, re-run Section 0 and Section 1, then start this cell again; finished stages can be skipped — run `status()` to see which.

In [ ]:
import pickle, time, numpy as np, pandas as pd

df, _ = load_and_audit(XLSX, GCB)
FEATS = candidate_features(df)
GDP, PE = "GDP (Constant LCU)", "Primary energy consumption (TWH)"
H = 6
space = model_space(families=["OLS", "Ridge", "Lasso", "ElasticNet"])
rows, sel_log = [], []
t0 = time.time()
for T in range(2000, 2019):
    tr = df.loc[:T]
    Xtr, ytr = tr[FEATS], tr[TARGET].values
    years = [y for y in range(T + 1, T + H + 1) if y <= 2024]
    Xte, yte = df.loc[years, FEATS], df.loc[years, TARGET].values
    tab = tune(Xtr, ytr, space)
    best = best_by_family(tab)
    prep = Prep().fit(Xtr)
    Hist = imputed_history(prep, Xtr)
    Xproj = project_features(Hist, len(years))
    gdp_proj = Xproj[GDP].values
    fc = {}
    for fam in space:
        b = best.loc[fam]
        fm = FittedModel(space[fam][int(b.cfg)], int(b.k)).fit(Xtr, ytr)
        fc[(fam, "ex post")] = fm.predict(Xte)
        fc[(fam, "ex ante")] = fm.predict(Xproj)
        sel_log.append(dict(origin=T, family=fam, k=len(fm.cols), params=describe(space[fam][int(b.cfg)]),
                            features="; ".join(SHORT.get(c, c) for c in fm.cols)))
    n = len(years)
    fc[("RW-drift", "ex ante")] = rw_drift(ytr, n)
    fc[("ARIMA", "ex ante")] = arima_auto(ytr, n)
    fc[("ETS (Holt)", "ex ante")] = ets_holt(ytr, n)
    Hte = imputed_history(prep, Xte)
    fc[("STIRPAT", "ex post")] = stirpat_fit_predict(Hist, ytr, Hte)
    fc[("STIRPAT", "ex ante")] = stirpat_fit_predict(Hist, ytr, Xproj)
    ex_tr = np.log(Hist[[GDP, PE]].values)
    fc[("ARIMAX", "ex post")] = arima_auto(ytr, n, ex_tr, np.log(Hte[[GDP, PE]].values))
    fc[("ARIMAX", "ex ante")] = arima_auto(ytr, n, ex_tr, np.log(Xproj[[GDP, PE]].values))
    scale = np.mean(np.abs(np.diff(ytr)))
    for (m, mode), p in fc.items():
        for i, yr in enumerate(years):
            rows.append(dict(origin=T, year=yr, h=i + 1, model=m, mode=mode, actual=yte[i], forecast=p[i],
                             ape=abs(yte[i] - p[i]) / yte[i] * 100, ae_scaled=abs(yte[i] - p[i]) / scale,
                             log_err=np.log(yte[i] / p[i]),
                             gdp_actual=df.loc[yr, GDP], gdp_proj=gdp_proj[i]))
    print(T, round(time.time() - t0, 1), flush=True)

R = pd.DataFrame(rows)
R.to_csv("out/rolling_origin_forecasts.csv", index=False)
pd.DataFrame(sel_log).to_csv("out/Table_S2_selection_by_origin.csv", index=False)
summ = R.groupby(["model", "mode", "h"]).agg(MAPE=("ape", "mean"), MASE=("ae_scaled", "mean"), n=("ape", "size")).reset_index()
summ.to_csv("out/Table3_rolling_origin_by_horizon.csv", index=False)
pd.set_option("display.width", 250)
print(summ.pivot_table(index=["model", "mode"], columns="h", values="MAPE").round(2))


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2000 80.3


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2001 140.2


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


...

2016 1026.6


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2017 1087.0


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2018 1148.6
h                      1     2     3     4      5      6
model      mode                                         
ARIMA      ex ante  1.95  3.34  4.65  5.78   6.64   7.30
ARIMAX     ex ante  3.14  4.97  6.62  7.90   9.17  10.37
           ex post  1.27  1.82  2.33  2.67   2.82   2.98
ETS (Holt) ex ante  2.43  4.08  5.79  7.25   9.00  10.54
ElasticNet ex ante  3.36  5.23  7.07  8.61  10.04  11.62
           ex post  1.65  2.79  3.83  4.50   5.12   5.29
Lasso      ex ante  3.48  5.45  7.42  8.92  10.50  11.89
           ex post  1.47  2.70  3.81  4.64   5.36   5.19
OLS        ex ante  3.33  5.30  7.07  8.46   9.95  11.15
           ex post  1.97  2.76  3.47  4.01   4.58   5.18
RW-drift   ex ante  1.95  3.34  4.65  5.78   6.64   7.30
Ridge      ex ante  3.41  5.38  7.19  9.00  10.63  12.38
           ex post  1.96  2.60  2.76  3.22   3.85   4.52
STIRPAT    ex ante  2.83  4.37  5.81  7.08   8.17   9.23
           ex post  1.83  2.11  2.35  2.61   2.83   2.95


## 4. Stage 3 — Stationary-target experiment  [R2-4, R4-4]
Every learner is re-tuned to predict annual log-changes; test predictions are re-integrated from the 2012 level.
Tree ensembles recover here, which identifies extrapolation (not tuning) as the reason they fail in levels.
Runtime ≈ 2 min.

In [ ]:
import pickle, time, numpy as np, pandas as pd

df, _ = load_and_audit(XLSX, GCB)
FEATS = candidate_features(df)
tr_years = df.loc[1965:2012].index
prep = Prep().fit(df.loc[tr_years, FEATS][eligible(df.loc[tr_years, FEATS])])
cols = prep.cols
Hall = imputed_history(prep, df[cols])       # imputer fitted on 1965-2012 only
D = pd.DataFrame(index=df.index[1:])
for c in cols:
    s = Hall[c]
    D[c] = np.diff(np.log(s.values)) if (s > 0).all() else np.diff(s.values)
dy = pd.Series(np.diff(np.log(df[TARGET].values)), index=df.index[1:])

Xtr, ytr = D.loc[1966:2012], dy.loc[1966:2012].values
Xte, yte = D.loc[2013:2024], dy.loc[2013:2024].values
t0 = time.time()
space = model_space()
tab = tune(Xtr, ytr, space, score="mae")
best = best_by_family(tab)
rows = []
y2012 = df.loc[2012, TARGET]
lev_te = df.loc[2013:2024, TARGET].values
lev_tr = df.loc[1965:2012, TARGET].values
for fam in space:
    b = best.loc[fam]
    fm = FittedModel(space[fam][int(b.cfg)], int(b.k)).fit(Xtr, ytr)
    d_hat = fm.predict(Xte)
    lev_hat = y2012 * np.exp(np.cumsum(d_hat))
    m = metrics(lev_te, lev_hat, lev_tr)
    rows.append(dict(family=fam, k=len(fm.cols), params=describe(space[fam][int(b.cfg)]),
                     R2_growth=r2_score(yte, d_hat), **{f"{kk}_level": v for kk, v in m.items()}))
    print(fam, round(time.time() - t0, 1), flush=True)
# benchmark: random walk with drift in logs
rw = y2012 * np.exp(np.mean(ytr) * np.arange(1, 13))
rows.append(dict(family="RW-drift", k=np.nan, params="", R2_growth=np.nan,
                 **{f"{kk}_level": v for kk, v in metrics(lev_te, rw, lev_tr).items()}))
res = pd.DataFrame(rows)
res.to_csv("out/Table_S4b_stationary_target_experiment.csv", index=False)
pd.set_option("display.width", 250)
print(res.round(4).to_string())


OLS 323.7
Ridge 328.1
Lasso 334.1
ElasticNet 338.3
RandomForest 342.7
ExtraTrees 348.8
GradientBoosting 353.2
XGBoost 357.5
LightGBM 363.5
CatBoost 368.0
SVR 372.2
MLP 378.6
              family     k                                                                      params  R2_growth  R2_level     MAE_level    RMSE_level  MAPE_level  MASE_level
0                OLS   8.0                                                                                -0.8509    0.5170  1.600509e+08  2.389300e+08      5.8485      4.2421
1              Ridge   8.0                                                   alpha=0.31622776601683794    -0.7889    0.5468  1.550734e+08  2.314439e+08      5.7011      4.1101
2              Lasso   8.0                                                 alpha=0.0031622776601683794    -0.7726    0.5468  1.552400e+08  2.314616e+08      5.7030      4.1145
3         ElasticNet   8.0                                                    alpha=0.01, l1_ratio=0.2    -0.7444    0.563

## 5. Stage 4 — Projection 2025–2030, NDC intensity benchmark, intervals and scenarios  [R2-6]
Equal-weight combination of the four linear learners (weights not derived from test performance), no smoothing,
empirical prediction intervals from the ex-ante errors of Stage 2, and Kaya-identity scenarios. Runtime ≈ 1 min.

In [ ]:
import pickle, numpy as np, pandas as pd

df, _ = load_and_audit(XLSX, GCB)
FEATS = candidate_features(df)
GDP, PE, POP = "GDP (Constant LCU)", "Primary energy consumption (TWH)", "Population"
LIN = ["OLS", "Ridge", "Lasso", "ElasticNet"]
space = model_space(families=LIN)
X, y = df[FEATS], df[TARGET].values
FUT = np.arange(2025, 2031)

tab = tune(X, y, space)
best = best_by_family(tab)
prep = Prep().fit(X)
Hist = imputed_history(prep, X)
Xf = project_features(Hist, 6)
fc, fitted, spec = {}, {}, []
for fam in LIN:
    b = best.loc[fam]
    fm = FittedModel(space[fam][int(b.cfg)], int(b.k)).fit(X, y)
    fc[fam] = fm.predict(Xf); fitted[fam] = fm
    spec.append(dict(family=fam, k=len(fm.cols), params=describe(space[fam][int(b.cfg)]), inner_cv_mape=b.cv_mape,
                     features="; ".join(SHORT.get(c, c) for c in fm.cols)))
spec = pd.DataFrame(spec); spec.to_csv("out/Table_production_specification.csv", index=False)
comb = np.mean([fc[f] for f in LIN], axis=0)

# ---- empirical prediction intervals from rolling-origin ex-ante errors -----------------------
R = pd.read_csv("out/rolling_origin_forecasts.csv")
ea = R[(R["mode"] == "ex ante") & (R.model.isin(LIN))]
combo = ea.groupby(["origin", "year", "h"]).agg(actual=("actual", "first"), forecast=("forecast", "mean"),
                                                   gdp_actual=("gdp_actual", "first"), gdp_proj=("gdp_proj", "first")).reset_index()
combo["log_err"] = np.log(combo.actual / combo.forecast)
combo["log_err_int"] = np.log((combo.actual / combo.gdp_actual) / (combo.forecast / combo.gdp_proj))
q = combo.groupby("h").log_err.quantile([0.05, 0.5, 0.95]).unstack()
qi = combo.groupby("h").log_err_int.quantile([0.05, 0.5, 0.95]).unstack()
combo.to_csv("out/rolling_combination_ex_ante.csv", index=False)

# ---- fossil-CO2 intensity benchmark ---------------------------------------------------------
gdp = df[GDP]; co2 = df[TARGET]
I05 = co2[2005] / gdp[2005]
gdp_f = Xf[GDP].values
red = lambda c, g: (1 - (c / g) / I05) * 100
red_2024 = red(co2[2024], gdp[2024]); red_2020 = red(co2[2020], gdp[2020])
red_2030 = red(comb[-1], gdp_f[-1])
# interval for 2030 intensity reduction from empirical h=6 intensity log errors
lo_I, hi_I = np.exp(qi.loc[6, 0.05]), np.exp(qi.loc[6, 0.95])
red_2030_lo, red_2030_hi = red(comb[-1] * hi_I, gdp_f[-1]), red(comb[-1] * lo_I, gdp_f[-1])
target_I = I05 * 0.55
abs_gap = comb[-1] - target_I * gdp_f[-1]
# required annual decline of intensity 2024->2030 vs recent trend
I24 = co2[2024] / gdp[2024]
req_rate = (1 - (target_I / I24) ** (1 / 6)) * 100
Iser = (co2 / gdp).loc[2014:2024]
trend_rate = (1 - np.exp(np.polyfit(Iser.index, np.log(Iser.values), 1)[0])) * 100

fore = pd.DataFrame({"Year": FUT, **{f"{f}_Mt": fc[f] / 1e6 for f in LIN}, "Combination_Mt": comb / 1e6,
                     "PI90_lo_Mt": comb * np.exp(q[0.05].values) / 1e6, "PI90_hi_Mt": comb * np.exp(q[0.95].values) / 1e6,
                     "GDP_projected": gdp_f, "Intensity_reduction_vs_2005_%": red(comb, gdp_f)})
fore.to_csv("out/Table4_production_forecast_2025_2030.csv", index=False)

# ---- Kaya-identity scenarios ------------------------------------------------------------------
E = df[PE]
EI, CI = E / gdp, co2 / E
def trend_g(s, a=2014, b=2024):
    s = s.loc[a:b]; return np.exp(np.polyfit(s.index, np.log(s.values), 1)[0]) - 1
gG, gEI, gCI = trend_g(gdp), trend_g(EI), trend_g(CI)
def kaya(dEI=0.0, dCI=0.0, dG=0.0, gG_=gG, gEI_=gEI, gCI_=gCI):
    t = np.arange(1, 7)
    G = gdp[2024] * (1 + gG_ + dG) ** t
    C = G * EI[2024] * (1 + gEI_ + dEI) ** t * CI[2024] * (1 + gCI_ + dCI) ** t
    return C, G
scen_def = {"S1 Trend continuation (Kaya)": dict(),
            "S2 Accelerated clean energy (CO2/E -1 pp/yr)": dict(dCI=-0.01),
            "S3 Enhanced efficiency (E/GDP -1 pp/yr)": dict(dEI=-0.01),
            "S4 Combined transition (S2+S3)": dict(dCI=-0.01, dEI=-0.01),
            "S5 Deep transition (CO2/E and E/GDP -2 pp/yr each)": dict(dCI=-0.02, dEI=-0.02)}
# bootstrap uncertainty of the trend growth rates (moving-block bootstrap of annual log-changes 2005-2024)
rng = np.random.default_rng(SEED)
# residuals around the 2014-2024 log-linear trends, resampled in blocks and re-fitted (trend uncertainty)
yrs = np.arange(2014, 2025)
comp = {"G": np.log(gdp.loc[2014:2024].values), "EI": np.log(EI.loc[2014:2024].values), "CI": np.log(CI.loc[2014:2024].values)}
fits = {k: np.polyfit(yrs, v, 1) for k, v in comp.items()}
res_ = {k: v - np.polyval(fits[k], yrs) for k, v in comp.items()}
B, L = 2000, 3
boot = []
for _ in range(B):
    starts = rng.integers(0, len(yrs) - L + 1, size=int(np.ceil(len(yrs) / L)))
    idx = np.concatenate([np.arange(s, s + L) for s in starts])[:len(yrs)]
    g = [np.exp(np.polyfit(yrs, np.polyval(fits[k], yrs) + res_[k][idx], 1)[0]) - 1 for k in ("G", "EI", "CI")]
    boot.append(g)
boot = np.array(boot)
srows = []
for name, d in scen_def.items():
    C, G = kaya(**d)
    reds = []
    for gG_b, gEI_b, gCI_b in boot:
        Cb, Gb = kaya(gG_=gG_b, gEI_=gEI_b, gCI_=gCI_b, **d)
        reds.append(red(Cb[-1], Gb[-1]))
    srows.append(dict(scenario=name, CO2_2030_Mt=C[-1] / 1e6, reduction_2030_pct=red(C[-1], G[-1]),
                      reduction_PI90_lo=np.percentile(reds, 5), reduction_PI90_hi=np.percentile(reds, 95),
                      gap_pp=red(C[-1], G[-1]) - 45, abs_gap_Mt=(C[-1] - target_I * G[-1]) / 1e6))
# GDP sensitivity on ML trend-continuation
for dg in (-0.01, 0.01):
    Xg = Xf.copy(); t = np.arange(1, 7)
    Xg[GDP] = Xf[GDP].values * (1 + dg) ** t
    cg = np.mean([fitted[f].predict(Xg) for f in LIN], axis=0)
    srows.append(dict(scenario=f"ML trend continuation, GDP growth {dg*100:+.0f} pp/yr", CO2_2030_Mt=cg[-1] / 1e6,
                      reduction_2030_pct=red(cg[-1], Xg[GDP].values[-1]), reduction_PI90_lo=np.nan,
                      reduction_PI90_hi=np.nan, gap_pp=red(cg[-1], Xg[GDP].values[-1]) - 45,
                      abs_gap_Mt=(cg[-1] - target_I * Xg[GDP].values[-1]) / 1e6))
scen = pd.DataFrame(srows); scen.to_csv("out/Table5_scenarios.csv", index=False)

# benchmark production forecasts (same GDP projection for intensity)
bn = {"RW-drift": rw_drift(y, 6), "ARIMA": arima_auto(y, 6), "STIRPAT (ex ante)": stirpat_fit_predict(Hist, y, Xf)}
bench_prod = pd.DataFrame([dict(method=k, CO2_2030_Mt=v[-1] / 1e6, reduction_2030_pct=red(v[-1], gdp_f[-1])) for k, v in bn.items()]
                          + [dict(method="Linear combination (ML)", CO2_2030_Mt=comb[-1] / 1e6, reduction_2030_pct=red_2030)])
bench_prod.to_csv("out/Table_S5_benchmark_production.csv", index=False)
# ex-ante accuracy for the INTENSITY metric at h=6 (rolling origins), ML combination vs benchmarks
ri = []
for m in ["RW-drift", "ARIMA", "STIRPAT", "ETS (Holt)"]:
    e = R[(R.model == m) & (R["mode"] == "ex ante") & (R.h == 6)]
    ri.append(dict(model=m, intensity_MAPE_h6=np.mean(np.abs((e.forecast / e.gdp_proj) / (e.actual / e.gdp_actual) - 1)) * 100,
                   co2_MAPE_h6=e.ape.mean()))
e = combo[combo.h == 6]
ri.append(dict(model="Linear combination (ML)", intensity_MAPE_h6=np.mean(np.abs((e.forecast / e.gdp_proj) / (e.actual / e.gdp_actual) - 1)) * 100,
               co2_MAPE_h6=np.mean(np.abs(e.forecast / e.actual - 1)) * 100))
ri = pd.DataFrame(ri); ri.to_csv("out/Table_S5b_intensity_ex_ante_accuracy.csv", index=False)
print(bench_prod.round(2)); print(ri.round(2))

# alternative feature-projection methods (sensitivity)
alt = []
for meth, w in (("loglinear", 10), ("loglinear", 15), ("loglinear", 20), ("quadratic", 30)):
    Xa = project_features(Hist, 6, window=w, method=meth)
    ca = np.mean([fitted[f].predict(Xa) for f in LIN], axis=0)
    alt.append(dict(method=f"{meth} ({w}y)", CO2_2030_Mt=ca[-1] / 1e6, reduction_2030_pct=red(ca[-1], Xa[GDP].values[-1])))
alt = pd.DataFrame(alt); alt.to_csv("out/Table_S6_projection_method_sensitivity.csv", index=False)

summary = dict(I05=I05, red_2020=red_2020, red_2024=red_2024, red_2030=red_2030, red_2030_PI90=(red_2030_lo, red_2030_hi),
               gap_pp=red_2030 - 45, abs_gap_Mt=abs_gap / 1e6, req_rate=req_rate, trend_rate=trend_rate,
               co2_2024_Mt=co2[2024] / 1e6, co2_2030_Mt=comb[-1] / 1e6, growth_2024_2030=(comb[-1] / co2[2024] - 1) * 100,
               gG=gG, gEI=gEI, gCI=gCI, q=q, qi=qi, n_origins_h6=int((combo.h == 6).sum()))
pickle.dump(dict(summary=summary, fore=fore, scen=scen, spec=spec, alt=alt, fitted=fitted, Xf=Xf, Hist=Hist,
                 tab=tab, fc=fc), open("out/stage4.pkl", "wb"))
pd.set_option("display.width", 250)
print(spec.to_string()); print(fore.round(2).to_string()); print(scen.round(2).to_string()); print(alt.round(2))
for k, v in summary.items():
    if k not in ("q", "qi"):
        print(k, v)
print(q); print(qi)


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarni

                    method  CO2_2030_Mt  reduction_2030_pct
0                 RW-drift      4347.67               12.20
1                    ARIMA      4347.54               12.20
2        STIRPAT (ex ante)      3831.27               22.62
3  Linear combination (ML)      3935.25               20.52
                     model  intensity_MAPE_h6  co2_MAPE_h6
0                 RW-drift               3.55         7.30
1                    ARIMA               3.55         7.30
2                  STIRPAT               6.01         9.23
3               ETS (Holt)               6.50        10.54
4  Linear combination (ML)               7.52        11.70
       family   k                     params  inner_cv_mape                                                                                                                                                                                                                                                                      features
0         OLS   

## 6. Stage 5 — Importance under collinearity, LMDI, structural breaks, cointegration  [R2-5, R4-1, R4-3, R4-5]
VIFs, block-bootstrap standardised coefficients, dominance analysis (Shapley R²) in levels and growth rates,
grouped permutation importance, LMDI decomposition, Chow/CUSUM tests, unit roots and cointegration,
post-2015 and emission-factor sensitivity. Runtime ≈ 2–3 min.

In [ ]:
import pickle, itertools, numpy as np, pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import breaks_cusumolsresid
import statsmodels.api as sm

df, _ = load_and_audit(XLSX, GCB)
FEATS = candidate_features(df)
S4 = pickle.load(open("out/stage4.pkl", "rb"))
ridge = S4["fitted"]["Ridge"]
cols = ridge.cols                                 # production feature set (shared by OLS/Ridge/EN)
alpha = ridge.cfg.alpha
GDP, PE, POP = "GDP (Constant LCU)", "Primary energy consumption (TWH)", "Population"
y = df[TARGET].values
Zfull = ridge.Z_train[cols]                       # standardised, imputed (fitted on 1965-2024)
out = {}

# ---- (a) collinearity diagnostics -------------------------------------------------------------
V = vif(Zfull); corr = Zfull.corr()
out["vif"] = V
# ---- (b) block-bootstrap standardised coefficients (Ridge) and Lasso selection frequency ------
rng = np.random.default_rng(SEED)
ys = (y - y.mean()) / y.std()
Lb = 5; n = len(y); coefs, sel = [], []
lasso_alpha = S4["fitted"]["Lasso"].cfg.alpha
for _ in range(1000):
    st = rng.integers(0, n - Lb + 1, size=int(np.ceil(n / Lb)))
    idx = np.concatenate([np.arange(s, s + Lb) for s in st])[:n]
    coefs.append(Ridge(alpha=alpha).fit(Zfull.values[idx], ys[idx]).coef_)
    sel.append(Lasso(alpha=lasso_alpha, max_iter=50000).fit(Zfull.values[idx], ys[idx]).coef_ != 0)
coefs = np.array(coefs); sel = np.array(sel)
point = Ridge(alpha=alpha).fit(Zfull.values, ys).coef_
coef_tab = pd.DataFrame({"feature": [SHORT[c] for c in cols], "sector": [SECTOR[c] for c in cols],
                         "std_coef": point, "CI95_lo": np.percentile(coefs, 2.5, 0), "CI95_hi": np.percentile(coefs, 97.5, 0),
                         "share_positive_%": (coefs > 0).mean(0) * 100, "lasso_selection_%": sel.mean(0) * 100, "VIF": V.values})
coef_tab.to_csv("out/Table6a_coefficients_bootstrap_vif.csv", index=False)

# ---- (c) dominance analysis (Shapley R2), levels and annual log-changes -----------------------
groups = {}
for c in cols:
    groups.setdefault(SECTOR[c], []).append(c)
phi_lvl, r2_lvl = shapley_r2(Zfull, y)
g_lvl, _ = shapley_r2(Zfull, y, groups)
H = S4["Hist"][cols]
Dx = pd.DataFrame({c: np.diff(np.log(H[c].values)) for c in cols}, index=H.index[1:])
Dy = np.diff(np.log(y))
Dz = (Dx - Dx.mean()) / Dx.std()
phi_d, r2_d = shapley_r2(Dz, Dy)
g_d, _ = shapley_r2(Dz, Dy, groups)
dom = pd.DataFrame({"levels_R2_share_%": phi_lvl / r2_lvl * 100, "growth_R2_share_%": phi_d / r2_d * 100})
dom.index = [SHORT[c] for c in dom.index]
gdom = pd.DataFrame({"levels_R2_share_%": g_lvl / r2_lvl * 100, "growth_R2_share_%": g_d / r2_d * 100})
dom.to_csv("out/Table6b_dominance_features.csv"); gdom.to_csv("out/Table6c_dominance_sectors.csv")
out.update(r2_lvl=r2_lvl, r2_growth=r2_d)

# ---- (d) grouped permutation importance on the 2013-2024 hold-out ----------------------------
Xtr, ytr = df.loc[1965:2012, FEATS], df.loc[1965:2012, TARGET].values
Xte, yte = df.loc[2013:2024, FEATS], df.loc[2013:2024, TARGET].values
fm = FittedModel(Ridge(alpha=alpha), len(cols), features=cols).fit(Xtr, ytr)
Zte = fm.prep.transform(Xte)[cols]
base = np.mean(np.abs(fm.model.predict(Zte.values) / yte - 1)) * 100
gp = {}
for gname, gcols in groups.items():
    inc = []
    for _ in range(500):
        Zp = Zte.copy(); perm = rng.permutation(len(Zp))
        Zp[gcols] = Zp[gcols].values[perm]
        inc.append(np.mean(np.abs(fm.model.predict(Zp.values) / yte - 1)) * 100 - base)
    gp[gname] = (np.mean(inc), np.percentile(inc, 5), np.percentile(inc, 95))
gperm = pd.DataFrame(gp, index=["mean_increase_MAPE_pp", "p5", "p95"]).T
gperm["holdout_MAPE_base"] = base
gperm.to_csv("out/Table6d_grouped_permutation.csv")

# ---- (e) LMDI Kaya decomposition ---------------------------------------------------------------
co2m = df[TARGET] / 1e6
lm = pd.DataFrame({f"{a}-{b}": lmdi_kaya(co2m, df[POP], df[GDP], df[PE], a, b)
                   for a, b in ((1990, 2005), (2005, 2024), (2014, 2024))})
lm.to_csv("out/Table7_LMDI_Kaya_MtCO2.csv")

# ---- (f) structural change: STIRPAT elasticities -------------------------------------------------
Hs = S4["Hist"]
Xs = pd.DataFrame(stirpat_design(Hs)[:, 1:], index=Hs.index, columns=["lnP", "lnA", "lnT_E/GDP", "lnF_fossil"])
ly = np.log(df[TARGET])
ols = sm.OLS(ly, sm.add_constant(Xs)).fit()
def chow(year):
    a, b = Xs.index < year, Xs.index >= year
    k = Xs.shape[1] + 1
    r = ols.ssr
    r1 = sm.OLS(ly[a], sm.add_constant(Xs[a])).fit().ssr
    r2 = sm.OLS(ly[b], sm.add_constant(Xs[b])).fit().ssr
    F = ((r - r1 - r2) / k) / ((r1 + r2) / (len(ly) - 2 * k))
    from scipy.stats import f as fdist
    return F, 1 - fdist.cdf(F, k, len(ly) - 2 * k)
def chow_pred(year):  # predictive-failure (Chow second) test
    a = Xs.index < year; n2 = (~a).sum(); k = Xs.shape[1] + 1
    r1 = sm.OLS(ly[a], sm.add_constant(Xs[a])).fit().ssr
    F = ((ols.ssr - r1) / n2) / (r1 / (a.sum() - k))
    from scipy.stats import f as fdist
    return F, 1 - fdist.cdf(F, n2, a.sum() - k)
cus = breaks_cusumolsresid(ols.resid.values, ddof=Xs.shape[1] + 1)
brk = pd.DataFrame([dict(test="Chow break 2015", stat=chow(2015)[0], p=chow(2015)[1]),
                    dict(test="Chow break 2016", stat=chow(2016)[0], p=chow(2016)[1]),
                    dict(test="Chow predictive failure 2015-2024", stat=chow_pred(2015)[0], p=chow_pred(2015)[1]),
                    dict(test="Chow predictive failure 2016-2024", stat=chow_pred(2016)[0], p=chow_pred(2016)[1]),
                    dict(test="OLS-CUSUM (Ploberger-Kramer)", stat=cus[0], p=cus[1]),
                    dict(test="Durbin-Watson (STIRPAT residuals)", stat=durbin_watson(ols.resid), p=np.nan)])
brk.to_csv("out/Table8_structural_break_tests.csv", index=False)
roll = []
for end in range(1990, 2025):
    w = (Xs.index > end - 25) & (Xs.index <= end)
    r = sm.OLS(ly[w], sm.add_constant(Xs[w])).fit()
    roll.append(dict(end=end, **r.params.drop("const").to_dict(),
                     **{f"se_{k}": v for k, v in r.bse.drop("const").to_dict().items()}))
roll = pd.DataFrame(roll); roll.to_csv("out/rolling_STIRPAT_elasticities.csv", index=False)
stir_full = pd.DataFrame({"coef": ols.params, "se": ols.bse}); stir_full.to_csv("out/STIRPAT_full_sample.csv")

# ---- (g) unit roots and cointegration ---------------------------------------------------------
series = {"ln CO2": ly.values, "ln GDP": np.log(Hs[GDP].values), "ln Primary energy": np.log(Hs[PE].values),
          "ln Population": np.log(Hs[POP].values),
          "ln Elec. cons. p.c.": np.log(Hs["Electric Power Consumption (kWh per capita)"].values)}
ur = []
for k, v in series.items():
    for lab, s in (("level", v), ("first difference", np.diff(v))):
        a = adfuller(s, regression="ct" if lab == "level" else "c", autolag="AIC")
        kp = kpss(s, regression="ct" if lab == "level" else "c", nlags="auto")
        ur.append(dict(series=k, form=lab, ADF_stat=a[0], ADF_p=a[1], KPSS_stat=kp[0], KPSS_p=kp[1]))
ur = pd.DataFrame(ur); ur.to_csv("out/Table_S3a_unit_roots.csv", index=False)
eg = coint(series["ln CO2"], np.column_stack([series["ln GDP"], series["ln Primary energy"], series["ln Population"]]), trend="c")
jo = coint_johansen(np.column_stack([series["ln CO2"], series["ln GDP"], series["ln Primary energy"]]), det_order=0, k_ar_diff=1)
co = pd.DataFrame([dict(test="Engle-Granger (ln CO2 on ln GDP, ln E, ln P)", stat=eg[0], p=eg[1], crit_5=eg[2][1]),
                   dict(test="Johansen trace r=0 (ln CO2, ln GDP, ln E)", stat=jo.lr1[0], p=np.nan, crit_5=jo.cvt[0, 1]),
                   dict(test="Johansen trace r<=1", stat=jo.lr1[1], p=np.nan, crit_5=jo.cvt[1, 1]),
                   dict(test="Johansen trace r<=2", stat=jo.lr1[2], p=np.nan, crit_5=jo.cvt[2, 1])])
co.to_csv("out/Table_S3b_cointegration.csv", index=False)

# ---- (h) post-2015 sensitivity of the production forecast --------------------------------------
LIN = ["OLS", "Ridge", "Lasso", "ElasticNet"]
Xf = S4["Xf"]; gdp_f = Xf[GDP].values; I05 = S4["summary"]["I05"]
red = lambda c, g: (1 - (c / g) / I05) * 100
sens = []
for start in (1990, 2000):
    sub = df.loc[start:]
    sp = model_space(families=LIN)
    tb = tune(sub[FEATS], sub[TARGET].values, sp); bb = best_by_family(tb)
    preds = [FittedModel(sp[f][int(bb.loc[f].cfg)], int(bb.loc[f].k)).fit(sub[FEATS], sub[TARGET].values).predict(Xf) for f in LIN]
    c = np.mean(preds, axis=0)
    sens.append(dict(variant=f"training window {start}-2024", CO2_2030_Mt=c[-1] / 1e6, reduction_2030_pct=red(c[-1], gdp_f[-1])))
# post-2015 level-shift dummy added to the production feature set
Xd = df[FEATS].copy(); Xd["post2015"] = (df.index >= 2015).astype(float)
Xfd = Xf.copy(); Xfd["post2015"] = 1.0
preds = []
for f in LIN:
    fmx = S4["fitted"][f]
    m = FittedModel(fmx.cfg, len(fmx.cols) + 1, features=fmx.cols + ["post2015"]).fit(Xd, y)
    preds.append(m.predict(Xfd))
c = np.mean(preds, axis=0)
sens.append(dict(variant="post-2015 level-shift indicator", CO2_2030_Mt=c[-1] / 1e6, reduction_2030_pct=red(c[-1], gdp_f[-1])))
# emission-factor sensitivity: carbon intensity of energy falls 1%/yr faster than implied by model
for dci in (-0.01, -0.02):
    cc = S4["fore"]["Combination_Mt"].values * 1e6 * (1 + dci) ** np.arange(1, 7)
    sens.append(dict(variant=f"CO2/E declines {abs(dci)*100:.0f} pp/yr faster (emission-factor change)",
                     CO2_2030_Mt=cc[-1] / 1e6, reduction_2030_pct=red(cc[-1], gdp_f[-1])))
sens = pd.DataFrame(sens); sens.to_csv("out/Table_S11_structural_sensitivity.csv", index=False)

pickle.dump(dict(coef_tab=coef_tab, dom=dom, gdom=gdom, gperm=gperm, lm=lm, brk=brk, roll=roll, ur=ur, co=co,
                 sens=sens, corr=corr, out=out, stir_full=stir_full), open("out/stage5.pkl", "wb"))
pd.set_option("display.width", 250)
for name, t in [("coef", coef_tab), ("dominance", dom), ("sector dominance", gdom), ("grouped perm", gperm), ("LMDI", lm),
                ("breaks", brk), ("unit roots", ur), ("coint", co), ("sens", sens), ("stirpat", stir_full)]:
    print("==", name); print(t.round(4).to_string())
print(out)
print(roll.round(3).iloc[::5].to_string())


/tmp/ipykernel_2112/157047034.py:126: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kp = kpss(s, regression="ct" if lab == "level" else "c", nlags="auto")
/tmp/ipykernel_2112/157047034.py:126: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kp = kpss(s, regression="ct" if lab == "level" else "c", nlags="auto")
/tmp/ipykernel_2112/157047034.py:126: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kp = kpss(s, regression="ct" if lab == "level" else "c", nlags="auto")
/tmp/ipykernel_2112/157047034.py:126: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is 

== coef
            feature    sector  std_coef  CI95_lo  CI95_hi  share_positive_%  lasso_selection_%       VIF
0  Elec. cons. p.c.    Energy    0.2349   0.1910   0.3045             100.0               99.7  417.3109
1           Imports  Economic    0.1133   0.0303   0.2028             100.0               82.3   78.4201
2        Population    Social    0.1306   0.0791   0.2101             100.0               98.8   25.8454
3               GDP  Economic    0.2985   0.2065   0.3494             100.0               99.9  189.2457
4   Energy use p.c.    Energy    0.1573   0.0697   0.2565             100.0               78.9  343.7091
5  Manufacturing VA  Economic    0.0387  -0.0279   0.0990              84.7                8.7  143.8376
== dominance
                  levels_R2_share_%  growth_R2_share_%
Elec. cons. p.c.            17.3734             1.8220
Imports                     16.4433             8.0055
Population                  15.1819            10.0610
GDP                     

## 7. Stage 6 — Data availability, data quality and data vintage  [R1-1, R1-2]
Helper first, then the deterministic experiments and the Monte Carlo runs. Runtime ≈ 8–12 min.

> **Second long block (10–15 min).** The first of its code cells clears its own output tables before appending, so always re-run it from the top rather than only part of it.

In [ ]:
import pickle, numpy as np, pandas as pd

LIN = ["OLS", "Ridge", "Lasso", "ElasticNet"]
GDP, PE, POP = "GDP (Constant LCU)", "Primary energy consumption (TWH)", "Population"
S4 = pickle.load(open("out/stage4.pkl", "rb"))   # production specification from Section 5
SPEC = {f: (S4["fitted"][f].cfg, S4["fitted"][f].cols) for f in LIN}   # production spec (fixed-spec runs)
base_df, _ = load_and_audit(XLSX, GCB)
FEATS = candidate_features(base_df)


def evaluate(df, retune=False, spec=SPEC, feats=FEATS):
    tr, te = df.loc[:2012], df.loc[2013:2024]
    y_all = df[TARGET].values
    I05 = df.loc[2005, TARGET] / df.loc[2005, GDP]
    if not np.isfinite(I05):
        I05 = base_df.loc[2005, TARGET] / base_df.loc[2005, GDP]
    red = lambda c, g: (1 - (c / g) / I05) * 100
    if retune:
        sp = model_space(families=LIN)
        bh = best_by_family(tune(tr[feats], tr[TARGET].values, sp))
        bp = best_by_family(tune(df[feats], y_all, sp))
        hold = [FittedModel(sp[f][int(bh.loc[f].cfg)], int(bh.loc[f].k)) for f in LIN]
        prod = [FittedModel(sp[f][int(bp.loc[f].cfg)], int(bp.loc[f].k)) for f in LIN]
    else:
        hold = [FittedModel(spec[f][0], len(spec[f][1]), features=spec[f][1]) for f in LIN]
        prod = [FittedModel(spec[f][0], len(spec[f][1]), features=spec[f][1]) for f in LIN]
    ph = np.mean([m.fit(tr[feats], tr[TARGET].values).predict(te[feats]) for m in hold], axis=0)
    mape = np.mean(np.abs(ph / te[TARGET].values - 1)) * 100
    fitted = [m.fit(df[feats], y_all) for m in prod]
    Hist = imputed_history(fitted[0].prep, df[feats])
    Xf = project_features(Hist, 6)
    c = np.mean([m.predict(Xf) for m in fitted], axis=0)
    return mape, red(c[-1], Xf[GDP].values[-1]), c[-1] / 1e6


In [ ]:
# --- deterministic experiments -------------------------------------------------------------
import os
for f in ["out/Table_S7a_data_experiments_deterministic.csv", "out/mc_data_quality_draws.csv"]:
    if os.path.exists(f):
        os.remove(f)

def record(name, df, **kw):
    m, r, c = evaluate(df, **kw)
    row = pd.DataFrame([dict(experiment=name, holdout_MAPE=m, red_2030=r, co2_2030_Mt=c)])
    fn = "out/Table_S7a_data_experiments_deterministic.csv"
    row.to_csv(fn, mode="a", header=not os.path.exists(fn), index=False)
    print(row.round(2).to_string(header=False), flush=True)

def rebuild(d):   # rebuild derived features after editing inputs
    d["Renewable_Share"] = d["Electricity from renewables - TWh"] / d["Electricity generation - TWh"] * 100
    d["Energy_Efficiency"] = d[GDP] / d[PE]
    return d

record("Baseline (re-tuned pipeline)", base_df, retune=True)
record("Baseline (fixed production specification)", base_df)
for drop in SPEC["Ridge"][1]:                                   # feature ablation
    sp = {f: (SPEC[f][0], [c for c in SPEC[f][1] if c != drop]) for f in LIN}
    record(f"Ablation: without {SHORT[drop]}", base_df, spec=sp)
mini = [GDP, POP, PE]                                           # minimal, near-universally available set
sp = model_space(families=LIN)
bb = best_by_family(tune(base_df.loc[:2012, FEATS], base_df.loc[:2012, TARGET].values, sp, fixed_features=mini))
record("Minimal configuration (GDP, population, primary energy)", base_df,
       spec={f: (sp[f][int(bb.loc[f].cfg)], mini) for f in LIN})
for var in (GDP, PE):                                           # systematic misreporting from 2015
    for b in (-0.05, 0.05):
        d = base_df.copy(); d.loc[2015:, var] *= (1 + b)
        record(f"Bias {b*100:+.0f}% in {SHORT[var]} from 2015", rebuild(d))
for start in (1985, 1995):                                      # shorter records
    record(f"Shorter record: data from {start} only (re-tuned)", base_df.loc[start:], retune=True)
d0, _ = load_and_audit(XLSX, None, use_gcb_target=False, zeros_as_missing=False, drop_suspect=False)
record("Original data handling (zeros as values, placeholders kept)", d0, retune=True)
d1, _ = load_and_audit(XLSX, None, use_gcb_target=False)
record("Target: original series (2024 placeholder retained)", d1, retune=True)
en = pd.read_csv(ENERGY).set_index("year")
d2 = base_df.copy()
for mine, theirs in (("Electricity from fossil fuels - TWh", "fossil_electricity"),
                     ("Electricity from renewables - TWh", "renewables_electricity"),
                     ("Electricity generation - TWh", "electricity_generation")):
    d2[mine] = en[theirs].reindex(d2.index)
record("Electricity series from Ember/Energy Institute (OWID 2026)", rebuild(d2), retune=True)
display(pd.read_csv("out/Table_S7a_data_experiments_deterministic.csv").round(2))


0  Baseline (re-tuned pipeline)  3.35  20.52  3935.25
0  Baseline (fixed production specification)  3.39  20.52  3935.25
0  Ablation: without Elec. cons. p.c.  2.36  20.61  3931.0
0  Ablation: without Imports  4.42  21.4  3891.88
0  Ablation: without Population  4.43  20.61  3930.95
0  Ablation: without GDP  6.38  22.22  3851.33
0  Ablation: without Energy use p.c.  3.04  19.65  3978.8
0  Ablation: without Manufacturing VA  3.58  20.45  3939.09
0  Minimal configuration (GDP, population, primary energy)  2.05  20.38  3942.39
0  Bias -5% in GDP from 2015  2.26  16.33  3935.58
0  Bias +5% in GDP from 2015  4.75  24.39  3931.05
0  Bias -5% in Primary energy from 2015  2.83  20.49  3936.92
0  Bias +5% in Primary energy from 2015  3.96  20.8  3921.71
0  Shorter record: data from 1985 only (re-tuned)  7.3  22.75  3825.21
0  Shorter record: data from 1995 only (re-tuned)  5.79  23.05  3810.1
0  Original data handling (zeros as values, placeholders kept)  2.77  22.0  3859.69
0  Target: original

,experiment,holdout_MAPE,red_2030,co2_2030_Mt
0,Baseline (re-tuned pipeline),3.35,20.52,3935.25
1,Baseline (fixed production specification),3.39,20.52,3935.25
2,Ablation: without Elec. cons. p.c.,2.36,20.61,3931.00
3,Ablation: without Imports,4.42,21.40,3891.88
4,Ablation: without Population,4.43,20.61,3930.95
5,Ablation: without GDP,6.38,22.22,3851.33
6,Ablation: without Energy use p.c.,3.04,19.65,3978.80
7,Ablation: without Manufacturing VA,3.58,20.45,3939.09
8,"Minimal configuration (GDP, population, primar...",2.05,20.38,3942.39
9,Bias -5% in GDP from 2015,2.26,16.33,3935.58


In [18]:
# --- Monte Carlo: randomly missing values and random measurement error ----------------------
rng = np.random.default_rng(SEED)
used = sorted(set(sum([SPEC[f][1] for f in LIN], [])))
mc = []
for kind, levels in (("mask", (0.1, 0.2, 0.3)), ("noise", (0.02, 0.05, 0.10))):
    for lev in levels:
        for draw in range(20):            # increase to 100+ draws if time permits
            d = base_df.copy()
            if kind == "mask":
                M = rng.random((len(d), len(used))) < lev
                M[-1, :] = False          # keep the 2024 row (forecast origin)
                vals = d[used].values.copy(); vals[M] = np.nan; d[used] = vals
            else:
                d[used] = d[used].values * (1 + rng.normal(0, lev, size=(len(d), len(used))))
            m, r, c = evaluate(d)
            mc.append(dict(kind=kind, level=lev, draw=draw, holdout_MAPE=m, red_2030=r, co2_2030_Mt=c))
        print(kind, lev, flush=True)
mc = pd.DataFrame(mc); mc.to_csv("out/mc_data_quality_draws.csv", index=False)
summary_mc = mc.groupby(["kind", "level"]).agg(
    MAPE_mean=("holdout_MAPE", "mean"), MAPE_p95=("holdout_MAPE", lambda x: np.percentile(x, 95)),
    red_mean=("red_2030", "mean"), red_p5=("red_2030", lambda x: np.percentile(x, 5)),
    red_p95=("red_2030", lambda x: np.percentile(x, 95)))
summary_mc.to_csv("out/Table_S7b_data_quality_montecarlo.csv")
display(summary_mc.round(2))


mask 0.1
mask 0.2
mask 0.3
noise 0.02
noise 0.05
noise 0.1


MAPE_mean  MAPE_p95  red_mean  red_p5  red_p95
kind  level                                                
mask  0.10        8.21     15.50     21.05   20.38    21.56
      0.20      106.84    125.74     21.20   20.03    22.45
      0.30       12.18     20.44     21.32   20.01    23.30
noise 0.02        3.36      5.33     20.85   16.85    24.10
      0.05        4.18      6.24     18.67   10.29    26.29
      0.10        5.57      8.46     15.69   -3.85    32.11

## 8. Stage 7 — Figures
Figures 1–6 and S1 of the revised manuscript (300 dpi PNG in `out/`).

In [19]:
import pickle, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300,
                     "savefig.bbox": "tight", "font.family": "DejaVu Sans"})
df, _ = load_and_audit(XLSX, GCB)
S1 = pickle.load(open("out/stage1.pkl", "rb")); S4 = pickle.load(open("out/stage4.pkl", "rb")); S5 = pickle.load(open("out/stage5.pkl", "rb"))
yrs = df.index.values; co2 = df[TARGET].values / 1e6

# ---------------- Figure 1: revised framework --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6.2)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 10)
boxes = [
    (0.2, 8.6, 4.4, 1.1, "#e3f2ec", "1. Open data & audit\n24 candidate indicators, 1965-2024; zero/placeholder\nvalues set to missing; GCB 2025 fossil CO2 (t)"),
    (5.4, 8.6, 4.4, 1.1, "#e3f2ec", "2. Leakage-free pipeline (fitted per training window)\nscaling -> iterative imputation -> coverage rule (>=50%)\n-> consensus ranking (MI, RF-MDI, permutation) -> top-k"),
    (0.2, 6.5, 4.4, 1.4, "#ecebfa", "3. Nested time-series validation\ninner TimeSeriesSplit: k and hyper-parameters\n11 learners, scaled target, identical folds\nhold-out 2013-2024 used once"),
    (5.4, 6.5, 4.4, 1.4, "#ecebfa", "4. Benchmarks & forecast evaluation\nRW-drift, ARIMA, ETS, STIRPAT, ARIMAX\n19 rolling origins (2000-2018), h = 1-6\nex post vs ex ante; MAPE, MASE, skill"),
    (0.2, 4.2, 4.4, 1.6, "#ecebfa", "5. Diagnostics\nstationary-target experiment (trees)\nunit roots / cointegration; Chow, CUSUM\nrolling elasticities; post-2015 sensitivity"),
    (5.4, 4.2, 4.4, 1.6, "#ecebfa", "6. Importance under collinearity\nVIF; bootstrap standardised coefficients\ndominance (Shapley R2) by sector\ngrouped permutation; LMDI (Kaya)"),
    (0.2, 1.8, 4.4, 1.7, "#fbeee6", "7. Trend-continuation projection 2025-2030\nequal-weight linear combination (no smoothing)\nempirical 90% intervals from ex-ante errors\nmulti-method comparison"),
    (5.4, 1.8, 4.4, 1.7, "#fbeee6", "8. Fossil-CO2 intensity vs NDC 45% benchmark\nKaya scenarios S1-S5; required decline rate\nreconciliation with BUR-4 GHG intensity\ndata-availability & data-quality experiments"),
]
for x, y0, w, h, c, t in boxes:
    ax.add_patch(FancyBboxPatch((x, y0), w, h, boxstyle="round,pad=0.05", fc=c, ec="#555555", lw=0.8))
    ax.text(x + w / 2, y0 + h / 2, t, ha="center", va="center", fontsize=8.3)
for (x1, y1, x2, y2) in [(4.6, 9.15, 5.4, 9.15), (7.6, 8.6, 2.4, 7.9), (4.6, 7.2, 5.4, 7.2), (2.4, 6.5, 2.4, 5.8),
                         (7.6, 6.5, 7.6, 5.8), (2.4, 4.2, 2.4, 3.5), (7.6, 4.2, 7.6, 3.5), (4.6, 2.65, 5.4, 2.65)]:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#666666", lw=1))
ax.text(5, 0.9, "Open-data AI framework for independent validation of national fossil-CO2 trajectories", ha="center", fontsize=9, style="italic")
plt.savefig("out/Figure1_framework.png"); plt.close()

# ---------------- Figure 2: consensus ranking on training data ---------------------------------
rk = S1["rank_tr"].copy()
norm = rk[["MI", "RF_MDI", "Permutation"]] / rk[["MI", "RF_MDI", "Permutation"]].max()
fig, ax = plt.subplots(figsize=(8, 6.5))
idx = np.arange(len(rk))[::-1]
for j, (c, col) in enumerate(zip(norm.columns, ["#4c72b0", "#dd8452", "#55a868"])):
    ax.barh(idx + (j - 1) * 0.27, norm[c].values, height=0.27, color=col, label=c.replace("RF_MDI", "RF impurity"))
ax.set_yticks(idx); ax.set_yticklabels([f"{SHORT.get(c, c)} ({r:.1f})" for c, r in zip(rk.index, rk.mean_rank)], fontsize=8)
ax.set_xlabel("Normalised importance (training window 1965-2012 only); mean rank in brackets")
ax.legend(frameon=False, loc="lower right"); ax.set_title("Consensus feature ranking (eligible candidates, coverage >= 50%)", fontsize=10)
plt.savefig("out/Figure2_consensus_ranking.png"); plt.close()

# ---------------- Figure 3: hold-out and stationary-target experiment ---------------------------
res = S1["res"]; preds = S1["preds"]; st = pd.read_csv("out/Table_S4b_stationary_target_experiment.csv")
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.25, 1]))
ax = axs[0]; ax.plot(yrs, co2, "k-", lw=2, label="Observed (GCB 2025)")
te = np.arange(2013, 2025)
for (fam, tag), col in [(("OLS", "selected"), "#1f77b4"), (("Ridge", "selected"), "#2ca02c"), (("Lasso", "selected"), "#9467bd"),
                        (("MLP", "selected"), "#8c564b"), (("XGBoost", "selected"), "#d62728"), (("RandomForest", "selected"), "#ff7f0e"),
                        (("RW-drift", "benchmark"), "#7f7f7f")]:
    ax.plot(te, preds[(fam, tag)][1] / 1e6, "--", color=col, label=fam)
ax.axvline(2012.5, color="grey", lw=0.8, ls=":"); ax.set_xlim(1990, 2024.5); ax.set_ylabel("Fossil CO2 (Mt)")
ax.set_title("(a) Hold-out 2013-2024 (trained on 1965-2012, ex post)", fontsize=10, loc="left"); ax.legend(frameon=False, fontsize=8, ncol=2)
ax = axs[1]
lev = res[res.set == "selected"].set_index("family")["MAPE_test"]
fams = ["OLS", "Ridge", "Lasso", "ElasticNet", "RandomForest", "ExtraTrees", "GradientBoosting", "XGBoost", "LightGBM", "CatBoost", "SVR", "MLP"]
stt = st.set_index("family")["MAPE_level"]
x = np.arange(len(fams))
ax.bar(x - 0.2, lev.reindex(fams).values, 0.4, label="Target in levels", color="#c44e52")
ax.bar(x + 0.2, stt.reindex(fams).values, 0.4, label="Stationary target (annual log-change)", color="#4c72b0")
ax.axhline(res.loc[res.family == "RW-drift", "MAPE_test"].values[0], color="grey", ls="--", lw=1, label="RW-drift benchmark")
ax.set_xticks(x); ax.set_xticklabels(fams, rotation=45, ha="right", fontsize=8); ax.set_ylabel("Hold-out MAPE (%)")
ax.set_title("(b) Tree ensembles recover when the target is stationary", fontsize=10, loc="left"); ax.legend(frameon=False, fontsize=8)
plt.savefig("out/Figure3_holdout_and_stationary.png"); plt.close()

# ---------------- Figure 4: rolling-origin accuracy by horizon ---------------------------------
R = pd.read_csv("out/rolling_origin_forecasts.csv")
LIN = ["OLS", "Ridge", "Lasso", "ElasticNet"]
comb = R[R.model.isin(LIN)].groupby(["origin", "year", "h", "mode"]).agg(actual=("actual", "first"), forecast=("forecast", "mean")).reset_index()
comb["ape"] = abs(comb.actual - comb.forecast) / comb.actual * 100; comb["model"] = "Linear combination"
RR = pd.concat([R, comb])
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for (m, mode), sty in {("Linear combination", "ex post"): ("#1f77b4", "-o"), ("Linear combination", "ex ante"): ("#1f77b4", "--o"),
                       ("STIRPAT", "ex post"): ("#2ca02c", "-s"), ("STIRPAT", "ex ante"): ("#2ca02c", "--s"),
                       ("RW-drift", "ex ante"): ("#7f7f7f", "--^"), ("ETS (Holt)", "ex ante"): ("#ff7f0e", "--v")}.items():
    s = RR[(RR.model == m) & (RR["mode"] == mode)].groupby("h").ape.mean()
    ax.plot(s.index, s.values, sty[1], color=sty[0], label=f"{m} ({mode})", ms=4)
ax.set_xlabel("Forecast horizon (years)"); ax.set_ylabel("Mean absolute percentage error (%)")
ax.set_title("Rolling-origin evaluation, 19 origins 2000-2018 (pipeline re-tuned at every origin)", fontsize=9.5)
ax.legend(frameon=False, fontsize=8); plt.savefig("out/Figure4_rolling_origin.png"); plt.close()

# ---------------- Figure 5: forecast, intensity and scenarios ------------------------------------
fo = S4["fore"]; sm_ = S4["summary"]; sc = S4["scen"]
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axs[0]; ax.plot(yrs[yrs >= 1990], co2[yrs >= 1990], "k-", lw=2, label="Observed")
ax.plot(fo.Year, fo.Combination_Mt, "o-", color="#1f77b4", label="Linear combination (trend continuation)")
ax.fill_between(fo.Year, fo.PI90_lo_Mt, fo.PI90_hi_Mt, color="#1f77b4", alpha=0.18, label="Empirical 90% interval")
bp = pd.read_csv("out/Table_S5_benchmark_production.csv").set_index("method")
ax.scatter([2030] * 3, bp.loc[["RW-drift", "STIRPAT (ex ante)"], "CO2_2030_Mt"].tolist() + [sc.iloc[0].CO2_2030_Mt],
           marker="D", color=["#7f7f7f", "#2ca02c", "#9467bd"], zorder=5)
for lab, v, c in [("RW-drift", bp.loc["RW-drift", "CO2_2030_Mt"], "#7f7f7f"), ("STIRPAT", bp.loc["STIRPAT (ex ante)", "CO2_2030_Mt"], "#2ca02c"),
                  ("Kaya S1", sc.iloc[0].CO2_2030_Mt, "#9467bd")]:
    ax.text(2030.3, v, lab, color=c, fontsize=8, va="center")
ax.set_xlim(1990, 2032.5); ax.set_ylabel("Fossil CO2 (Mt)"); ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("(a) Fossil CO2: observed and 2025-2030 projection", fontsize=10, loc="left")
ax = axs[1]
I = (df[TARGET] / df["GDP (Constant LCU)"]); redh = (1 - I / I[2005]) * 100
ax.plot(redh.loc[2005:].index, redh.loc[2005:].values, "k-", lw=2, label="Observed fossil-CO2 intensity reduction")
ax.plot(fo.Year, fo["Intensity_reduction_vs_2005_%"], "o-", color="#1f77b4", label="ML trend continuation")
lo, hi = sm_["red_2030_PI90"]; ax.errorbar([2030.15], [sm_["red_2030"]], yerr=[[sm_["red_2030"] - lo], [hi - sm_["red_2030"]]], color="#1f77b4", capsize=4)
cols = ["#9467bd", "#8c564b", "#e377c2", "#bcbd22", "#17becf"]
for (i, r), c in zip(sc.iloc[:5].iterrows(), cols):
    ax.plot([2024, 2030], [redh[2024], r.reduction_2030_pct], ":", color=c, label=r.scenario)
ax.axhline(45, color="red", ls="--", lw=1.2, label="NDC benchmark: 45% below 2005")
ax.scatter([2020], [36], marker="*", s=120, color="red", zorder=5); ax.text(2012.5, 33.5, "BUR-4: total GHG intensity\n-36% by 2020 (not comparable)", fontsize=7.5, color="red")
ax.set_ylabel("Reduction vs 2005 (%)"); ax.set_xlim(2005, 2031); ax.set_ylim(-8, 50)
ax.legend(frameon=False, fontsize=6.8, loc="upper left"); ax.set_title("(b) Fossil-CO2 intensity vs NDC benchmark", fontsize=10, loc="left")
plt.savefig("out/Figure5_forecast_intensity_scenarios.png"); plt.close()

# ---------------- Figure 6: importance under collinearity and LMDI -----------------------------
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
g = S5["gdom"]; x = np.arange(len(g))
axs[0].bar(x - 0.2, g["levels_R2_share_%"], 0.4, label="Levels", color="#4c72b0")
axs[0].bar(x + 0.2, g["growth_R2_share_%"], 0.4, label="Annual log-changes", color="#dd8452")
axs[0].set_xticks(x); axs[0].set_xticklabels(g.index); axs[0].set_ylabel("Share of explained variance (%)")
axs[0].set_title("(a) Dominance analysis (Shapley R2) by sector", fontsize=10, loc="left"); axs[0].legend(frameon=False)
gp = S5["gperm"]
axs[1].bar(gp.index, gp.mean_increase_MAPE_pp, yerr=[gp.mean_increase_MAPE_pp - gp.p5, gp.p95 - gp.mean_increase_MAPE_pp], color="#55a868", capsize=4)
axs[1].set_ylabel("Increase in hold-out MAPE (pp)"); axs[1].set_title("(b) Grouped permutation importance (2013-2024)", fontsize=10, loc="left")
lm = S5["lm"].drop("Total change")
xx = np.arange(len(lm.columns)); w = 0.2
for j, (f, c) in enumerate(zip(lm.index, ["#8172b2", "#4c72b0", "#55a868", "#c44e52"])):
    axs[2].bar(xx + (j - 1.5) * w, lm.loc[f].values, w, label=f, color=c)
axs[2].axhline(0, color="k", lw=0.6); axs[2].set_xticks(xx); axs[2].set_xticklabels(lm.columns)
axs[2].set_ylabel("Contribution to change in fossil CO2 (Mt)"); axs[2].legend(frameon=False, fontsize=7.5)
axs[2].set_title("(c) LMDI decomposition (Kaya identity)", fontsize=10, loc="left")
plt.savefig("out/Figure6_importance_LMDI.png"); plt.close()

# ---------------- Figure S1: rolling STIRPAT elasticities ---------------------------------------
roll = S5["roll"]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for c, col in zip(["lnP", "lnA", "lnT_E/GDP", "lnF_fossil"], ["#8172b2", "#4c72b0", "#55a868", "#c44e52"]):
    ax.plot(roll.end, roll[c], color=col, label=c)
    ax.fill_between(roll.end, roll[c] - 1.96 * roll[f"se_{c}"], roll[c] + 1.96 * roll[f"se_{c}"], color=col, alpha=0.12)
ax.axvline(2015, color="grey", ls=":"); ax.set_xlabel("End year of 25-year window"); ax.set_ylabel("Elasticity")
ax.set_title("Rolling STIRPAT elasticities (95% CI)", fontsize=10); ax.legend(frameon=False, fontsize=8)
plt.savefig("out/FigureS1_rolling_elasticities.png"); plt.close()
print("figures done")


figures done


In [ ]:
for f in ["Figure1_framework", "Figure2_consensus_ranking", "Figure3_holdout_and_stationary",
          "Figure4_rolling_origin", "Figure5_forecast_intensity_scenarios", "Figure6_importance_LMDI",
          "FigureS1_rolling_elasticities"]:
    display(Image(f"out/{f}.png", width=900))


Output hidden; open in https://colab.research.google.com to view.

## 9. Outputs

| File | Content |
|---|---|
| `Table_S1_data_audit.csv` | every data correction, with reason |
| `Table1_consensus_ranking_train1965_2012.csv` | consensus ranking on the training window |
| `Table2_holdout_all_models.csv` | hold-out performance of all learners and benchmarks |
| `Table3_rolling_origin_by_horizon.csv`, `rolling_origin_forecasts.csv` | rolling-origin results |
| `Table_S4b_stationary_target_experiment.csv` | stationary-target experiment |
| `Table4_production_forecast_2025_2030.csv`, `Table5_scenarios.csv`, `Table_S5*`, `Table_S6*` | projection, scenarios, benchmarks, projection-method sensitivity |
| `Table6a–6d`, `Table7_LMDI_Kaya_MtCO2.csv`, `Table8_structural_break_tests.csv`, `Table_S3a/b`, `Table_S11` | importance, decomposition, diagnostics |
| `Table_S7a/b` | data-availability and data-quality experiments |
| `Figure1–6`, `FigureS1` (PNG) | manuscript figures |

Before resubmission, refresh the indicator file where values were placeholders (electricity consumption per
capita 2023–24, primary energy production from 2013, energy intensity 2023–24, HDI 2024) and re-run this
notebook; all numbers in the manuscript should then be taken from the regenerated tables.

## 10. Download the results
Zips everything in `out/` (tables + figures). The zip is also left in the project folder on Drive, so you can fetch it later from Drive instead.

In [20]:
import shutil
status()
zip_path = shutil.make_archive("JCLEPRO_revision_outputs", "zip", "out")
print("Created:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)


,status
section,
2 Stage 1 hold-out evaluation,done
3 Stage 2 rolling origins,done
4 Stage 3 stationary target,done
5 Stage 4 projection and scenarios,done
6 Stage 5 importance and diagnostics,done
7 Stage 6 data experiments,done
8 Stage 7 figures,done


Created: /content/drive/MyDrive/JCLEPRO_revision/JCLEPRO_revision_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>